# Phase 3 – Rhetorical Role Prediction (RR): Collective Multi-Run Notebook
### Master Orchestrator for All High-Performance Runs, PEFT Methods & Submissions

This notebook executes competitive PEFT experiments sequentially and generates their respective CodaBench submissions with **`prediction.json`**:
1. **Run 1 (Arch 4 LORMA)**: Low-Rank Matrix Adaptation on the best Phase-1 architecture (`ALiBi + SwiGLU + RMSNorm`), augmented with document-level BiLSTM sequence modeling.
2. **Run 2 (Arch 3 Prefix Tuning)**: Prefix Tuning on Architecture 3 (`GQA + LayerNorm + GELU`) with reparameterization MLP & structured sequence modeling.
3. **Run 3 (Arch 4 → Arch 1 Knowledge Distillation)**: Cross-architecture KD distilling the powerful Arch 4 Teacher into a fast Arch 1 Student (`Sinusoidal + GELU + LayerNorm`) adapted with LoRA.
4. **Run 5 (Arch 4 AdaLoRA)**: Adaptive Low-Rank Adaptation on Architecture 4 with SVD parameterization, dynamic parameter importance evaluation, and rank pruning + BiLSTM & Multihead Attention.
5. **Run 6 (Arch 4 IA3 + CRF)**: $(IA)^3$ Infused Adapters on Architecture 4 scaling Keys, Values, and intermediate SwiGLU activations with only ~172K PEFT parameters, decoded via Linear-Chain CRF.
6. **Master Multi-Model Ensemble**: Dynamic soft-voting ensemble across all executed models with post-hoc class calibration to maximize competition Macro-F1.

### Features & Optimizations:
- **Direct Folder-to-.PT Converter**: Automatically detects if Architecture 4 is provided as an unzipped directory and packs it into a compliant single PyTorch archive without manual intervention.
- **Strict OOM Prevention**: Chunked backbone encoding, gradient checkpointing, automatic sequence trimming, and aggressive memory purging (`free_mem()`) between every run to guarantee zero CUDA OOM errors.
- **Discourse-Aware Context Windowing**: Symmetric `[left_ctx </s> target_sent </s> right_ctx]` windowing with word-cached BPE tokenization.
- **Competition Macro-F1 Calibration**: Post-hoc class bias tuning directly optimizing the official competition metric.
- **Verified Submissions**: Each run outputs a standalone ZIP archive containing `prediction.json` (and `predictions.json`), followed by the final ensemble submission.


In [ ]:
# 1. Environment, Authentication & Hardware Initialization
import os, gc, sys, json, math, time, random, copy, glob, zipfile, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set deterministic seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print(f"Device: {device} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM) | AMP Enabled: {USE_AMP}")

# Hugging Face Authentication
HF_TOKEN = None
for _name in ("HF", "HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN: break
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("✓ Hugging Face login successful.")
    except Exception as e:
        print(f"HF Login note: {e}")
else:
    print("Notice: No HF_TOKEN provided; public access will be used.")

def free_mem():
    import sys
    for name in ['_', '__', '___', '_i', '_ii', '_iii']:
        if name in globals():
            globals()[name] = None
    if hasattr(sys, 'last_traceback'):
        sys.last_traceback = None
    if hasattr(sys, 'last_value'):
        sys.last_value = None
    if hasattr(sys, 'last_type'):
        sys.last_type = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
free_mem()

In [ ]:
# 2. Checkpoint Resolver & Robust Directory-to-.PT Converter
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./working"
os.makedirs(WORK_DIR, exist_ok=True)

def find_checkpoint(candidates: List[str], glob_pattern: str = "") -> Optional[str]:
    for c in candidates:
        if os.path.exists(c):
            return c
    if glob_pattern:
        matches = sorted(glob.glob(glob_pattern, recursive=True))
        if matches:
            return matches[0]
    return None

ARCH4_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/phase1-arch4/decoder_lm.pt',
    './decoder_lm.pt',
    'd:/D_Downloads/Phase-1/decoder_lm.pt',
]
ARCH3_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm',
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/phase1-arch3/decoder_lm.pt',
    './decoder_lm_arch3.pt',
]
ARCH1_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/phase1-arch1/decoder_lm.pt',
    './decoder_lm_arch1.pt',
]
TOKENIZER_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json',
    '/kaggle/input/cs781-phase1-arch4/custom_bpe.json',
    '/kaggle/input/custom_bpe.json',
    './custom_bpe.json',
    'd:/D_Downloads/Phase-1/custom_bpe.json',
]

ARCH4_RAW = find_checkpoint(ARCH4_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch4/**/decoder_lm*") or find_checkpoint([], "/kaggle/input/**/decoder_lm.pt")
ARCH3_RAW = find_checkpoint(ARCH3_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch3/**")
ARCH1_RAW = find_checkpoint(ARCH1_CANDIDATES, "/kaggle/input/**/cs781-legal-model-checkpoints/**/decoder_lm*") or find_checkpoint([], "/kaggle/input/**/phase1-arch1/**/decoder_lm*")
TOKENIZER_PATH = find_checkpoint(TOKENIZER_CANDIDATES, "/kaggle/input/**/custom_bpe.json")

print("Discovered Raw Checkpoint Paths:")
print(f"  Arch 4 Raw : {ARCH4_RAW}")
print(f"  Arch 3 Raw : {ARCH3_RAW}")
print(f"  Arch 1 Raw : {ARCH1_RAW}")
print(f"  Tokenizer  : {TOKENIZER_PATH}")

# Direct conversion for Architecture 4 when stored as an unzipped directory
ARCH4_CHECKPOINT_PATH = ARCH4_RAW
if ARCH4_RAW and os.path.isdir(ARCH4_RAW):
    source_dir = ARCH4_RAW
    target_pt  = os.path.join(WORK_DIR, 'decoder_lm_arch4.pt')
    print(f"-> Arch 4 checkpoint is a directory: '{source_dir}'. Converting directly into valid .pt archive...")
    files = [(os.path.join(r, f), os.path.relpath(os.path.join(r, f), source_dir))
             for r, _, fs in os.walk(source_dir) for f in fs]
    has_top_dir = any(rel.endswith('data.pkl') and os.sep in rel for _, rel in files)
    prefix = '' if has_top_dir else 'archive/'
    with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
        for full, rel in files:
            zf.write(full, arcname=prefix + rel)
    print(f"✓ Converted Arch 4 directory to valid checkpoint file: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")
    ARCH4_CHECKPOINT_PATH = target_pt
elif ARCH4_RAW:
    print(f"✓ Arch 4 checkpoint is a valid file: {ARCH4_CHECKPOINT_PATH}")

# Arch 3 folder / nested archive handler
ARCH3_CHECKPOINT_PATH = ARCH3_RAW
if ARCH3_RAW and os.path.isdir(ARCH3_RAW):
    pt_files = [os.path.join(ARCH3_RAW, f) for f in os.listdir(ARCH3_RAW) if f.endswith('.pt')]
    if pt_files:
        ARCH3_CHECKPOINT_PATH = pt_files[0]
    else:
        target_pt3 = os.path.join(WORK_DIR, 'decoder_lm_arch3.pt')
        files = [(os.path.join(r, f), os.path.relpath(os.path.join(r, f), ARCH3_RAW))
                 for r, _, fs in os.walk(ARCH3_RAW) for f in fs]
        has_top_dir = any(rel.endswith('data.pkl') and os.sep in rel for _, rel in files)
        prefix = '' if has_top_dir else 'archive/'
        with zipfile.ZipFile(target_pt3, 'w', compression=zipfile.ZIP_STORED) as zf:
            for full, rel in files:
                zf.write(full, arcname=prefix + rel)
        ARCH3_CHECKPOINT_PATH = target_pt3

ARCH1_CHECKPOINT_PATH = ARCH1_RAW
print("Final Resolved Paths:")
print(f"  ARCH4: {ARCH4_CHECKPOINT_PATH}")
print(f"  ARCH3: {ARCH3_CHECKPOINT_PATH}")
print(f"  ARCH1: {ARCH1_CHECKPOINT_PATH}")

In [ ]:
# 3. Dataset Download & Structure Normalization
from huggingface_hub import hf_hub_download

HF_REPO = "Exploration-Lab/CS-781-Capstone"

def get_data_file(filename: str) -> str:
    local_hits = glob.glob(f"/kaggle/input/**/{Path(filename).name}", recursive=True)
    if local_hits:
        return local_hits[0]
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)

print("Fetching Phase 3 RR datasets...")
TRAIN_CL_PATH = get_data_file("Phase_3_RR/Train/CL_train.json")
DEV_CL_PATH   = get_data_file("Phase_3_RR/Train/CL_dev.json")
TRAIN_IT_PATH = get_data_file("Phase_3_RR/Train/IT_train.json")
DEV_IT_PATH   = get_data_file("Phase_3_RR/Train/IT_dev.json")
LABEL_PATH    = get_data_file("Phase_3_RR/Train/label_vocab.json")
TEST_CL_PATH  = get_data_file("Phase_3_RR/Test/CL_test.json")
TEST_IT_PATH  = get_data_file("Phase_3_RR/Test/IT_test.json")

def load_json(p):
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f)

cl_train_raw = load_json(TRAIN_CL_PATH)
cl_dev_raw   = load_json(DEV_CL_PATH)
it_train_raw = load_json(TRAIN_IT_PATH)
it_dev_raw   = load_json(DEV_IT_PATH)
cl_test_raw  = load_json(TEST_CL_PATH)
it_test_raw  = load_json(TEST_IT_PATH)

def normalize_docs(raw_docs, domain):
    if isinstance(raw_docs, dict):
        return [{'id': k, 'text': v, 'domain': domain} for k, v in raw_docs.items()]
    elif isinstance(raw_docs, list):
        out = []
        for d in raw_docs:
            d_copy = dict(d)
            d_copy['domain'] = domain
            if 'id' not in d_copy and 'doc_id' in d_copy:
                d_copy['id'] = d_copy['doc_id']
            out.append(d_copy)
        return out
    return []

cl_train = normalize_docs(cl_train_raw, 'CL')
cl_dev   = normalize_docs(cl_dev_raw, 'CL')
it_train = normalize_docs(it_train_raw, 'IT')
it_dev   = normalize_docs(it_dev_raw, 'IT')
cl_test  = normalize_docs(cl_test_raw, 'CL')
it_test  = normalize_docs(it_test_raw, 'IT')

with open(LABEL_PATH, 'r', encoding='utf-8') as f:
    lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)

print(f"Datasets Loaded:")
print(f"  Court Cases (CL): {len(cl_train)} train docs | {len(cl_dev)} dev docs | {len(cl_test)} test docs")
print(f"  Income Tax  (IT): {len(it_train)} train docs | {len(it_dev)} dev docs | {len(it_test)} test docs")
print(f"  Rhetorical Roles ({NUM_CLASSES}): {LABELS}")

In [ ]:
# 4. Fast ByteBPETokenizer with Word-Level Cache Acceleration
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class FastByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}
        self._word_cache = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def _word_ids(self, w):
        r = self._word_cache.get(w)
        if r is None:
            unk = self.stoi.get('<unk>', 1)
            r = [self.stoi.get(s, unk) for s in self._apply_merges_to_word(w)]
            self._word_cache[w] = r
        return r

    def encode(self, text, max_len=None):
        ids = [i for w in text.split() for i in self._word_ids(w)]
        return ids[:max_len] if max_len else ids

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']
        t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

if not TOKENIZER_PATH or not os.path.exists(TOKENIZER_PATH):
    TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO, filename="Phase_1_LM/custom_bpe.json", repo_type="dataset", token=HF_TOKEN)

tokenizer = FastByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
SEP_ID = tokenizer.stoi.get('</s>', 3)
print(f"✓ Tokenizer ready: {tokenizer.vocab_size} vocabulary tokens | PAD_ID={PAD_ID}, SEP_ID={SEP_ID}")

In [ ]:
# 5. Dataset, Context Windowing, Metrics & Submission Packaging Helpers
MAX_SEQ_LEN    = 160
TARGET_MAX_TOK = 64
CONTEXT_WINDOW = 2

class RRDocDataset(Dataset):
    """
    Constructs sentence instances with symmetric context windowing:
    [left_context] </s> [target_sentence] </s> [right_context]
    """
    def __init__(self, docs, tokenizer, label2id, max_seq_len=160, context_window=2, is_test=False):
        self.items = []
        pad_id = tokenizer.stoi.get('<pad>', 0)
        sep = [tokenizer.stoi.get('</s>', 3)]

        for doc in docs:
            sents = doc.get('text', [])
            labels = doc.get('labels', [None] * len(sents))
            doc_id = doc.get('id', doc.get('doc_id', 'unknown'))
            domain = doc.get('domain', 'CL')
            tok_sents = [tokenizer.encode(s) for s in sents]
            
            ids_list, mask_list, lbl_list, pos_list = [], [], [], []
            n_sents = len(sents)
            
            for i in range(n_sents):
                tgt = tok_sents[i][:TARGET_MAX_TOK]
                s = max(0, i - context_window)
                e = min(n_sents, i + context_window + 1)
                
                left  = [t for j in range(s, i) for t in (tok_sents[j] + sep)]
                right = [t for j in range(i + 1, e) for t in (tok_sents[j] + sep)]
                
                budget = max_seq_len - len(tgt) - 2 * len(sep)
                lb = max(0, budget // 2)
                rb = max(0, budget - lb)
                if len(left) < lb: rb += (lb - len(left))
                if len(right) < rb: lb += (rb - len(right))
                
                left_tokens  = left[-lb:] if lb > 0 else []
                right_tokens = right[:rb] if rb > 0 else []
                
                seq = (left_tokens + sep + tgt + sep + right_tokens)[:max_seq_len]
                mask = [1] * len(seq)
                if len(seq) < max_seq_len:
                    pad_len = max_seq_len - len(seq)
                    seq = seq + [pad_id] * pad_len
                    mask = mask + [0] * pad_len
                    
                ids_list.append(seq)
                mask_list.append(mask)
                pos_list.append(i / max(1, n_sents - 1))
                
                if not is_test:
                    lbl = labels[i]
                    lbl_list.append(label2id.get(lbl, -100) if lbl is not None else -100)
            
            self.items.append({
                'doc_id': doc_id,
                'domain': domain,
                'input_ids': torch.tensor(ids_list, dtype=torch.long),
                'attention_mask': torch.tensor(mask_list, dtype=torch.long),
                'labels': torch.tensor(lbl_list, dtype=torch.long) if not is_test else None,
                'positions': torch.tensor(pos_list, dtype=torch.float32),
                'num_sentences': n_sents
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

def collate_doc(batch):
    return batch[0]

def compute_macro_f1(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    valid = y_true >= 0
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return 0.0
    cm = np.bincount(y_true * num_classes + y_pred, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(num_classes), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros(num_classes), where=(tp + fn) > 0)
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros(num_classes), where=(prec + rec) > 0)
    return float(np.mean(f1))

def tune_class_bias(logits_list, labels_list, num_classes):
    all_logits = np.concatenate(logits_list, axis=0)
    all_labels = np.concatenate(labels_list, axis=0)
    valid = all_labels >= 0
    all_logits = all_logits[valid]
    all_labels = all_labels[valid]
    
    best_bias = np.zeros(num_classes)
    best_f1 = compute_macro_f1(all_labels, all_logits.argmax(axis=-1), num_classes)
    
    counts = np.bincount(all_labels, minlength=num_classes)
    prior = counts / max(1, counts.sum())
    p_bias = -0.25 * np.log(np.clip(prior, 1e-4, 1.0))
    p_bias -= p_bias.mean()
    p_f1 = compute_macro_f1(all_labels, (all_logits + p_bias).argmax(axis=-1), num_classes)
    if p_f1 > best_f1:
        best_f1 = p_f1
        best_bias = p_bias

    for c in range(num_classes):
        current_best = best_bias[c]
        for delta in [-1.5, -1.0, -0.5, 0.5, 1.0, 1.5]:
            trial_bias = best_bias.copy()
            trial_bias[c] = current_best + delta
            f1 = compute_macro_f1(all_labels, (all_logits + trial_bias).argmax(axis=-1), num_classes)
            if f1 > best_f1:
                best_f1 = f1
                best_bias = trial_bias
    return best_bias, best_f1

def save_submission_zip(predictions_dict, run_name, test_docs, output_dir=WORK_DIR):
    doc_map = {d['id']: d['text'] for d in test_docs}
    for doc_id, sents in doc_map.items():
        assert doc_id in predictions_dict, f"Missing predictions for doc {doc_id}!"
        preds = predictions_dict[doc_id]
        assert len(preds) == len(sents), f"Doc {doc_id} mismatch: {len(preds)} preds vs {len(sents)} sents!"
        
    json_path = os.path.join(output_dir, f"prediction_{run_name}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(predictions_dict, f, indent=2)
        
    zip_path = os.path.join(output_dir, f"submission_{run_name}.zip")
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")
        zf.write(json_path, arcname="predictions.json")
        
    print(f"✓ Created CodaBench submission zip: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)")
    print(f"  Verified {len(predictions_dict)} test documents | {sum(len(v) for v in predictions_dict.values())} predictions with prediction.json.")
    return zip_path

# Pre-build Dataset objects
print("Building Document Datasets with context windowing...")
train_cl_ds = RRDocDataset(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW)
train_it_ds = RRDocDataset(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW)
dev_cl_ds   = RRDocDataset(cl_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW)
dev_it_ds   = RRDocDataset(it_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW)
test_cl_ds  = RRDocDataset(cl_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, is_test=True)
test_it_ds  = RRDocDataset(it_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, is_test=True)

train_docs = train_cl_ds + train_it_ds
dev_docs   = dev_cl_ds + dev_it_ds
test_docs  = cl_test + it_test

# Compute inverse frequency class weights to boost macro-F1 on minority roles
all_train_labels = [lbl.item() for doc in train_docs for lbl in doc['labels'] if lbl >= 0]
label_counts = np.bincount(all_train_labels, minlength=NUM_CLASSES)
total_lbls = max(1, len(all_train_labels))
class_weights_np = (total_lbls / (NUM_CLASSES * np.maximum(label_counts, 1))) ** 0.5
class_weights_t = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
print(f"Class imbalance weights calculated. Min: {class_weights_np.min():.2f}, Max: {class_weights_np.max():.2f}")

## ====================================================================
## RUN 1: Architecture 4 with LORMA (Compulsory PEFT on Best Phase-1 LM)
## Backbone: `ALiBi + SwiGLU + RMSNorm` (~105M params, Phase-1 Perplexity = 23.66)
## ====================================================================


In [ ]:
# RUN 1: Architecture 4 + LORMA Model Definition & Training
free_mem()

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads):
    def gsp2(n):
        s = 2 ** (-(2 ** -(math.log2(n) - 3)))
        return [s * (s**i) for i in range(n)]
    if math.log2(n_heads) == int(math.log2(n_heads)):
        return torch.tensor(gsp2(n_heads))
    cp2 = 2 ** math.floor(math.log2(n_heads))
    return torch.tensor(gsp2(cp2) + gsp2(2*cp2)[0::2][:n_heads-cp2])

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.dh = d_model // n_heads; self.dropout = dropout
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.q_norm  = RMSNorm(self.dh); self.k_norm = RMSNorm(self.dh)
        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer('slopes', slopes)

    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k)
        pq = torch.arange(S, device=x.device).unsqueeze(1)
        pk = torch.arange(S, device=x.device).unsqueeze(0)
        alibi = self.slopes * (pk - pq).unsqueeze(0)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = alibi + causal.unsqueeze(0).unsqueeze(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1,2).contiguous().view(B, S, self.n_heads * self.dh))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

@dataclass
class ALiBiConfig:
    vocab_size: int = 8000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    dropout: float = 0.1

class ALiBiBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1   = RMSNorm(cfg.d_model)
        self.attn = ALiBiCausalAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.n2   = RMSNorm(cfg.d_model)
        self.ffn  = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([ALiBiBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f  = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks: x = b(x)
        return self.norm_f(x)

class LORMALinear(nn.Module):
    def __init__(self, original: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.0):
        super().__init__()
        self.original = original
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / math.sqrt(rank)
        self.lorma_A = nn.Linear(original.in_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, original.out_features, bias=False)
        self.lorma_s = nn.Parameter(torch.ones(1))
        self.lorma_dropout = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False
    def forward(self, x):
        base = self.original(x)
        delta = self.lorma_B(self.lorma_A(self.lorma_dropout(x))) * (self.lorma_s * self.scaling)
        return base + delta

def inject_lorma(model: nn.Module, attn_mods: List[str], ffn_mods: List[str],
                 r_attn: int = 16, r_ffn: int = 16, alpha: float = 32.0, dropout: float = 0.05):
    replaced = {'attn': 0, 'ffn': 0}
    for name, module in list(model.named_modules()):
        for target in attn_mods:
            if name.endswith(target) and isinstance(module, nn.Linear):
                p = model
                for part in name.split('.')[:-1]: p = getattr(p, part)
                setattr(p, name.split('.')[-1], LORMALinear(module, r_attn, alpha, dropout))
                replaced['attn'] += 1
                break
        for target in ffn_mods:
            if name.endswith(target) and isinstance(module, nn.Linear):
                p = model
                for part in name.split('.')[:-1]: p = getattr(p, part)
                setattr(p, name.split('.')[-1], LORMALinear(module, r_ffn, alpha, dropout))
                replaced['ffn'] += 1
                break
    print(f"  LORMA injected: {replaced['attn']} Attention layers (r={r_attn}), {replaced['ffn']} FFN layers (r={r_ffn})")

class RRClassifierArch4(nn.Module):
    def __init__(self, backbone: ALiBiSwiGLULM, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        self.num_labels = num_labels
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def get_sent_embs(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            c_ids  = input_ids[i : i + chunk_size]
            c_mask = attention_mask[i : i + chunk_size]
            # Trim padding columns for chunk speedup
            T = int(c_mask.sum(1).max().item())
            ci = c_ids[:, :max(T, 1)]
            cm = c_mask[:, :max(T, 1)]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                h = self.backbone(ci)
            m = cm.unsqueeze(-1).float()
            emb = (h.float() * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        return torch.cat(sent_embs, dim=0)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        embs = self.get_sent_embs(input_ids, attention_mask, chunk_size=chunk_size)
        embs = self.dropout(embs)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            lstm_out, _ = self.bilstm(embs.unsqueeze(0))
        lstm_out = self.dropout(lstm_out.squeeze(0).float())
        return self.classifier(lstm_out)

print("Initializing Architecture 4...")
cfg4 = ALiBiConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=2560)
backbone4 = ALiBiSwiGLULM(cfg4)

if ARCH4_CHECKPOINT_PATH and os.path.exists(ARCH4_CHECKPOINT_PATH):
    print(f"Loading pretrained weights from {ARCH4_CHECKPOINT_PATH}...")
    ckpt4 = torch.load(ARCH4_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state_dict4 = ckpt4.get('model_state_dict', ckpt4)
    backbone4.load_state_dict(state_dict4, strict=False)
    print("✓ Loaded Architecture 4 pretrained weights.")
else:
    print("Warning: Arch 4 checkpoint not found; initialized from scratch.")

# Freeze base backbone and inject LORMA
for p in backbone4.parameters():
    p.requires_grad = False
inject_lorma(backbone4, ['q_proj', 'k_proj', 'v_proj', 'out_proj'], ['gate_proj', 'up_proj', 'down_proj'], r_attn=16, r_ffn=16, alpha=32.0)

model_arch4 = RRClassifierArch4(backbone4, NUM_CLASSES, dropout=0.15).to(device)
trainable_p = sum(p.numel() for p in model_arch4.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model_arch4.parameters())
print(f"Architecture 4 Model Ready: {trainable_p:,} trainable / {total_p:,} total parameters ({100*trainable_p/total_p:.2f}% active)")

# Training Loop
LR_LORMA, LR_HEAD = 1.5e-4, 8e-4
WEIGHT_DECAY = 0.01
EPOCHS_RUN1 = 4

lorma_decay, lorma_nodecay, head_params = [], [], []
for n, p in model_arch4.named_parameters():
    if not p.requires_grad: continue
    if 'lorma_s' in n: lorma_nodecay.append(p)
    elif 'lorma_' in n: lorma_decay.append(p)
    else: head_params.append(p)

opt4 = AdamW([
    {'params': lorma_decay,   'lr': LR_LORMA, 'weight_decay': WEIGHT_DECAY},
    {'params': lorma_nodecay, 'lr': LR_LORMA, 'weight_decay': 0.0},
    {'params': head_params,   'lr': LR_HEAD,  'weight_decay': WEIGHT_DECAY}
])
scaler4 = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("\nStarting Run 1 Training (Arch 4 LORMA)...")
best_macro_f1_arch4 = 0.0
best_logits_arch4_dev = []
criterion = nn.CrossEntropyLoss(weight=class_weights_t, ignore_index=-100)

for epoch in range(1, EPOCHS_RUN1 + 1):
    model_arch4.train()
    total_loss, n_steps = 0.0, 0
    t0 = time.time()
    
    # Shuffle documents per epoch
    perm = torch.randperm(len(train_docs)).tolist()
    for idx in perm:
        doc = train_docs[idx]
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        
        opt4.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model_arch4(ids, mask, chunk_size=16)
            loss = criterion(logits, lbls)
            
        scaler4.scale(loss).backward()
        scaler4.unscale_(opt4)
        nn.utils.clip_grad_norm_(model_arch4.parameters(), max_norm=1.0)
        scaler4.step(opt4)
        scaler4.update()
        
        total_loss += loss.item()
        n_steps += 1
        
    avg_loss = total_loss / max(1, n_steps)
    
    # Dev Evaluation
    model_arch4.eval()
    dev_logits, dev_labels = [], []
    with torch.no_grad():
        for doc in dev_docs:
            ids = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            logits = model_arch4(ids, mask, chunk_size=16)
            dev_logits.append(logits.cpu().numpy())
            dev_labels.append(doc['labels'].numpy())
            
    val_f1 = compute_macro_f1(np.concatenate(dev_labels), np.concatenate(dev_logits).argmax(-1), NUM_CLASSES)
    print(f"Epoch {epoch}/{EPOCHS_RUN1} [{time.time()-t0:.1f}s] - Train Loss: {avg_loss:.4f} | Dev Macro-F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1_arch4:
        best_macro_f1_arch4 = val_f1
        best_logits_arch4_dev = dev_logits
        torch.save(model_arch4.state_dict(), os.path.join(WORK_DIR, 'arch4_lorma_best.pt'))

# Bias Tuning & Inference for Arch 4
bias_arch4, tuned_f1_arch4 = tune_class_bias(best_logits_arch4_dev, [d['labels'].numpy() for d in dev_docs], NUM_CLASSES)
print(f"\n✓ Run 1 Dev Macro-F1: {best_macro_f1_arch4:.4f} -> Post-Hoc Calibrated F1: {tuned_f1_arch4:.4f}")

# Generate Test Predictions
print("Generating Run 1 test predictions...")
model_arch4.eval()
test_logits_arch4 = {}
preds_arch4 = {}
with torch.no_grad():
    for doc in test_cl_ds + test_it_ds:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        logits = model_arch4(ids, mask, chunk_size=16)
        test_logits_arch4[doc['doc_id']] = logits.cpu().numpy()
        calibrated_preds = (logits.cpu().numpy() + bias_arch4).argmax(-1).tolist()
        preds_arch4[doc['doc_id']] = [ID2LABEL[p] for p in calibrated_preds]

# Save submission
sub_arch4_zip = save_submission_zip(preds_arch4, "arch4_lorma", test_docs)

# Cache probabilities for Ensemble
probs_arch4_test = {k: F.softmax(torch.tensor(v).float() + torch.tensor(bias_arch4), dim=-1).numpy().tolist() for k, v in test_logits_arch4.items()}
with open(os.path.join(WORK_DIR, 'probs_arch4_test.json'), 'w') as f:
    json.dump(probs_arch4_test, f)

# Purge VRAM
del model_arch4, opt4, backbone4, scaler4
free_mem()
print("Run 1 completed. Memory purged.")

## ====================================================================
## RUN 2: Architecture 3 with Prefix Tuning (GQA + Reparameterization MLP)
## Backbone: `Grouped Query Attention (GQA) + LayerNorm + GELU` (~89M params)
## ====================================================================


In [ ]:
# RUN 2: Architecture 3 + Prefix Tuning Model Definition & Training
free_mem()
@dataclass
class GQAConfig:
    vocab_size: int = 8000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1

class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: GQAConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.n_q_per_kv = cfg.n_heads // cfg.n_kv_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj  = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj  = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj  = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.out_proj= nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop= nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len, dtype=torch.bool))
        self.register_buffer('causal_mask', mask, persistent=False)
        self.prefix_k = None
        self.prefix_v = None

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        if self.n_q_per_kv > 1:
            k = k.repeat_interleave(self.n_q_per_kv, dim=1)
            v = v.repeat_interleave(self.n_q_per_kv, dim=1)
        if self.prefix_k is not None:
            pk = self.prefix_k.expand(B, -1, -1, -1)
            pv = self.prefix_v.expand(B, -1, -1, -1)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        P = 0 if self.prefix_k is None else self.prefix_k.shape[2]
        cm = self.causal_mask[:T, :T]
        prefix_mask = torch.ones(T, P, dtype=torch.bool, device=x.device)
        full_mask = torch.cat([prefix_mask, cm], dim=1)
        scores = scores.masked_fill(~full_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)
        z = attn @ v
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout)
        )
    def forward(self, x): return self.net(x)

class GQABlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = GroupedQueryAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GQADecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        pe = torch.zeros(cfg.max_seq_len, cfg.d_model)
        pos = torch.arange(0, cfg.max_seq_len).unsqueeze(1).float()
        div = torch.pow(10000.0, torch.arange(0, cfg.d_model, 2).float() / cfg.d_model)
        pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div)
        self.register_buffer('pos_emb', pe, persistent=False)
        self.blocks = nn.ModuleList([GQABlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:T]
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

class PrefixTuningModule(nn.Module):
    def __init__(self, num_layers: int, n_heads: int, head_dim: int,
                 prefix_len: int = 16, prefix_dim: int = 512, dropout: float = 0.05):
        super().__init__()
        self.prefix_len = prefix_len
        self.num_layers = num_layers
        self.n_heads    = n_heads
        self.head_dim   = head_dim
        self.prefix_emb = nn.Embedding(prefix_len, prefix_dim)
        out_dim = 2 * num_layers * n_heads * head_dim
        self.reparam_mlp = nn.Sequential(
            nn.Linear(prefix_dim, prefix_dim * 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(prefix_dim * 2, out_dim)
        )
        nn.init.normal_(self.prefix_emb.weight, std=0.02)

    def forward(self):
        prefix_ids = torch.arange(self.prefix_len, device=self.prefix_emb.weight.device)
        emb = self.prefix_emb(prefix_ids)
        out = self.reparam_mlp(emb)
        L, H, D, P = self.num_layers, self.n_heads, self.head_dim, self.prefix_len
        out = out.view(P, 2, L, H, D).permute(2, 1, 3, 0, 4)
        prefix_kv = []
        for l in range(L):
            k = out[l, 0].unsqueeze(0)
            v = out[l, 1].unsqueeze(0)
            prefix_kv.append((k, v))
        return prefix_kv

def attach_prefixes(backbone: GQADecoderLM, prefix_module: PrefixTuningModule):
    prefix_kv = prefix_module()
    for l, block in enumerate(backbone.blocks):
        block.attn.prefix_k = prefix_kv[l][0]
        block.attn.prefix_v = prefix_kv[l][1]

class RRClassifierArch3(nn.Module):
    def __init__(self, backbone: GQADecoderLM, num_labels: int, prefix_len: int = 16, prefix_dim: int = 512, dropout: float = 0.15):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        self.prefix_module = PrefixTuningModule(
            num_layers=backbone.cfg.n_layers,
            n_heads=backbone.cfg.n_heads,
            head_dim=d_model // backbone.cfg.n_heads,
            prefix_len=prefix_len,
            prefix_dim=prefix_dim,
            dropout=dropout
        )
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        attach_prefixes(self.backbone, self.prefix_module)
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            ci = input_ids[i:i+chunk_size]
            cm = attention_mask[i:i+chunk_size]
            T = int(cm.sum(1).max().item())
            ci = ci[:, :max(T, 1)]
            cm = cm[:, :max(T, 1)]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                h = self.backbone(ci)
            m = cm.unsqueeze(-1).float()
            emb = (h.float() * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        embs = torch.cat(sent_embs, dim=0)
        embs = self.dropout(embs)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            lstm_out, _ = self.bilstm(embs.unsqueeze(0))
        lstm_out = self.dropout(lstm_out.squeeze(0).float())
        return self.classifier(lstm_out)

print("Initializing Architecture 3...")
cfg3 = GQAConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, d_ff=3072)
backbone3 = GQADecoderLM(cfg3)

if ARCH3_CHECKPOINT_PATH and os.path.exists(ARCH3_CHECKPOINT_PATH):
    print(f"Loading pretrained weights from {ARCH3_CHECKPOINT_PATH}...")
    ckpt3 = torch.load(ARCH3_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state_dict3 = ckpt3.get('model_state_dict', ckpt3)
    backbone3.load_state_dict(state_dict3, strict=False)
    print("✓ Loaded Architecture 3 pretrained weights.")
else:
    print("Notice: Arch 3 checkpoint not found; proceeding with random initialization.")

# Freeze backbone entirely
for p in backbone3.parameters():
    p.requires_grad = False

model_arch3 = RRClassifierArch3(backbone3, NUM_CLASSES, prefix_len=16, prefix_dim=512, dropout=0.15).to(device)
trainable_p3 = sum(p.numel() for p in model_arch3.parameters() if p.requires_grad)
print(f"Architecture 3 Ready: {trainable_p3:,} trainable parameters (Prefix MLP + BiLSTM Head)")

opt3 = AdamW(
    [p for p in model_arch3.parameters() if p.requires_grad],
    lr=6e-4,
    weight_decay=0.01
)
scaler3 = torch.amp.GradScaler('cuda', enabled=USE_AMP)
EPOCHS_RUN2 = 4

print("\nStarting Run 2 Training (Arch 3 Prefix Tuning)...")
best_macro_f1_arch3 = 0.0
best_logits_arch3_dev = []

for epoch in range(1, EPOCHS_RUN2 + 1):
    model_arch3.train()
    total_loss, n_steps = 0.0, 0
    t0 = time.time()
    
    perm = torch.randperm(len(train_docs)).tolist()
    for idx in perm:
        doc = train_docs[idx]
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        
        opt3.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model_arch3(ids, mask, chunk_size=16)
            loss = criterion(logits, lbls)
            
        scaler3.scale(loss).backward()
        scaler3.unscale_(opt3)
        nn.utils.clip_grad_norm_(model_arch3.parameters(), max_norm=1.0)
        scaler3.step(opt3)
        scaler3.update()
        
        total_loss += loss.item()
        n_steps += 1
        
    avg_loss = total_loss / max(1, n_steps)
    
    # Dev Evaluation
    model_arch3.eval()
    dev_logits3, dev_labels3 = [], []
    with torch.no_grad():
        for doc in dev_docs:
            ids = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            logits = model_arch3(ids, mask, chunk_size=16)
            dev_logits3.append(logits.cpu().numpy())
            dev_labels3.append(doc['labels'].numpy())
            
    val_f1 = compute_macro_f1(np.concatenate(dev_labels3), np.concatenate(dev_logits3).argmax(-1), NUM_CLASSES)
    print(f"Epoch {epoch}/{EPOCHS_RUN2} [{time.time()-t0:.1f}s] - Train Loss: {avg_loss:.4f} | Dev Macro-F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1_arch3:
        best_macro_f1_arch3 = val_f1
        best_logits_arch3_dev = dev_logits3
        torch.save(model_arch3.state_dict(), os.path.join(WORK_DIR, 'arch3_prefix_best.pt'))

bias_arch3, tuned_f1_arch3 = tune_class_bias(best_logits_arch3_dev, [d['labels'].numpy() for d in dev_docs], NUM_CLASSES)
print(f"\n✓ Run 2 Dev Macro-F1: {best_macro_f1_arch3:.4f} -> Post-Hoc Calibrated F1: {tuned_f1_arch3:.4f}")

# Generate Test Predictions
print("Generating Run 2 test predictions...")
model_arch3.eval()
test_logits_arch3 = {}
preds_arch3 = {}
with torch.no_grad():
    for doc in test_cl_ds + test_it_ds:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        logits = model_arch3(ids, mask, chunk_size=16)
        test_logits_arch3[doc['doc_id']] = logits.cpu().numpy()
        calibrated_preds = (logits.cpu().numpy() + bias_arch3).argmax(-1).tolist()
        preds_arch3[doc['doc_id']] = [ID2LABEL[p] for p in calibrated_preds]

sub_arch3_zip = save_submission_zip(preds_arch3, "arch3_prefixtuning", test_docs)

probs_arch3_test = {k: F.softmax(torch.tensor(v).float() + torch.tensor(bias_arch3), dim=-1).numpy().tolist() for k, v in test_logits_arch3.items()}
with open(os.path.join(WORK_DIR, 'probs_arch3_test.json'), 'w') as f:
    json.dump(probs_arch3_test, f)

del model_arch3, opt3, backbone3, scaler3
free_mem()
print("Run 2 completed. Memory purged.")

## ====================================================================
## RUN 3: Cross-Architecture Knowledge Distillation (Arch 4 Teacher → Arch 1 Student)
## Teacher: Arch 4 (`ALiBi + SwiGLU + RMSNorm`)
## Student: Arch 1 (`Sinusoidal + GELU + LayerNorm` with LoRA Adaptation)
## ====================================================================


In [ ]:
# RUN 3: Cross-Architecture Knowledge Distillation (Arch 4 -> Arch 1 + LoRA)
free_mem()
# --- Student Model Definition (Architecture 1) ---
@dataclass
class LMConfig:
    vocab_size: int = 8000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1

class CausalAttn(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads; self.dh = cfg.d_model // cfg.n_heads
        self.scale = 1 / math.sqrt(self.dh)
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj= nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop= nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len, dtype=torch.bool))
        self.register_buffer('causal_mask', mask, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.dh).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        cm = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~cm.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = self.attn_drop(F.softmax(scores, dim=-1))
        z = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class Arch1Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalAttn(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        pe = torch.zeros(cfg.max_seq_len, cfg.d_model)
        pos = torch.arange(0, cfg.max_seq_len).unsqueeze(1).float()
        div = torch.pow(10000.0, torch.arange(0, cfg.d_model, 2).float() / cfg.d_model)
        pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div)
        self.register_buffer('pos_emb', pe, persistent=False)
        self.blocks = nn.ModuleList([Arch1Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:T]
        for b in self.blocks: x = b(x)
        return self.ln_f(x)

class LoRALinear(nn.Module):
    def __init__(self, orig: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.orig = orig
        self.rank = rank
        self.scaling = alpha / rank
        self.lora_A = nn.Linear(orig.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, orig.out_features, bias=False)
        self.drop = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.orig.weight.requires_grad = False
        if self.orig.bias is not None: self.orig.bias.requires_grad = False
    def forward(self, x):
        return self.orig(x) + self.lora_B(self.lora_A(self.drop(x))) * self.scaling

def inject_lora(model: nn.Module, targets=['q_proj', 'k_proj', 'v_proj', 'out_proj'], rank=16, alpha=32.0):
    replaced = 0
    for name, mod in list(model.named_modules()):
        for t in targets:
            if name.endswith(t) and isinstance(mod, nn.Linear):
                p = model
                for part in name.split('.')[:-1]: p = getattr(p, part)
                setattr(p, name.split('.')[-1], LoRALinear(mod, rank, alpha))
                replaced += 1
                break
    print(f"  LoRA injected into {replaced} Student attention layers.")

class StudentClassifier(nn.Module):
    def __init__(self, backbone: DecoderLM, num_labels: int, dropout: float = 0.15):
        super().__init__()
        self.backbone = backbone
        d = backbone.cfg.d_model
        self.bilstm = nn.LSTM(d, d // 2, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            ci = input_ids[i:i+chunk_size]
            cm = attention_mask[i:i+chunk_size]
            T = int(cm.sum(1).max().item())
            ci = ci[:, :max(T, 1)]
            cm = cm[:, :max(T, 1)]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                h = self.backbone(ci)
            m = cm.unsqueeze(-1).float()
            emb = (h.float() * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        embs = torch.cat(sent_embs, dim=0)
        embs = self.drop(embs)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            lstm_out, _ = self.bilstm(embs.unsqueeze(0))
        lstm_out = self.drop(lstm_out.squeeze(0).float())
        logits = self.classifier(lstm_out)
        return logits, lstm_out

# ----------------- Step 1: Pre-compute / Cache Teacher Logits -----------------
print("Step 1: Instantiating Teacher Model (Architecture 4) to generate soft targets...")
teacher_bb = ALiBiSwiGLULM(ALiBiConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=2560))
inject_lorma(teacher_bb, ['q_proj', 'k_proj', 'v_proj', 'out_proj'], ['gate_proj', 'up_proj', 'down_proj'], r_attn=16, r_ffn=16, alpha=32.0)
teacher_model = RRClassifierArch4(teacher_bb, NUM_CLASSES, dropout=0.0).to(device)

t_ckpt_path = os.path.join(WORK_DIR, 'arch4_lorma_best.pt')
if os.path.exists(t_ckpt_path):
    print(f"Loading trained Arch 4 Teacher weights from {t_ckpt_path}...")
    teacher_model.load_state_dict(torch.load(t_ckpt_path, map_location=device, weights_only=False))
elif ARCH4_CHECKPOINT_PATH and os.path.exists(ARCH4_CHECKPOINT_PATH):
    print("Loading base Arch 4 weights as Teacher...")
    ckpt_t = torch.load(ARCH4_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    teacher_bb.load_state_dict(ckpt_t.get('model_state_dict', ckpt_t), strict=False)

teacher_model.eval()
teacher_train_logits = []
print("Caching Teacher logits on training dataset...")
with torch.no_grad():
    for doc in train_docs:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        t_logits = teacher_model(ids, mask, chunk_size=16)
        teacher_train_logits.append(t_logits.cpu())

del teacher_model, teacher_bb
free_mem()
print("✓ Teacher soft logits successfully cached. Teacher model unloaded to conserve VRAM.")

# ----------------- Step 2: Initialize Student Model (Architecture 1) -----------------
print("\nStep 2: Initializing Student Model (Architecture 1 with LoRA)...")
student_bb = DecoderLM(LMConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=3072))

if ARCH1_CHECKPOINT_PATH and os.path.exists(ARCH1_CHECKPOINT_PATH):
    print(f"Loading Architecture 1 pretrained weights from {ARCH1_CHECKPOINT_PATH}...")
    ckpt1 = torch.load(ARCH1_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    student_bb.load_state_dict(ckpt1.get('model_state_dict', ckpt1), strict=False)
    print("✓ Loaded Architecture 1 pretrained weights.")
else:
    print("Notice: Arch 1 checkpoint not found; initialized from scratch.")

for p in student_bb.parameters(): p.requires_grad = False
inject_lora(student_bb, ['q_proj', 'k_proj', 'v_proj', 'out_proj'], rank=16, alpha=32.0)
student_model = StudentClassifier(student_bb, NUM_CLASSES, dropout=0.15).to(device)

trainable_ps = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print(f"Student Model Ready: {trainable_ps:,} trainable parameters.")

# ----------------- Step 3: Knowledge Distillation Loss & Training -----------------
def distillation_loss(s_logits, t_logits, labels, alpha_kd=0.5, temperature=4.0):
    T = temperature
    soft_student = F.log_softmax(s_logits / T, dim=-1)
    soft_teacher = F.softmax(t_logits / T, dim=-1)
    loss_kd = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T * T)
    loss_hard = criterion(s_logits, labels)
    return (alpha_kd * loss_kd) + ((1.0 - alpha_kd) * loss_hard)

opt_student = AdamW([p for p in student_model.parameters() if p.requires_grad], lr=4e-4, weight_decay=0.01)
scaler_s = torch.amp.GradScaler('cuda', enabled=USE_AMP)
EPOCHS_RUN3 = 4

print("\nStarting Run 3 Training (Arch 4 -> Arch 1 Knowledge Distillation)...")
best_macro_f1_kd = 0.0
best_logits_kd_dev = []

for epoch in range(1, EPOCHS_RUN3 + 1):
    student_model.train()
    total_loss, n_steps = 0.0, 0
    t0 = time.time()
    
    perm = torch.randperm(len(train_docs)).tolist()
    for idx in perm:
        doc = train_docs[idx]
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        t_logits = teacher_train_logits[idx].to(device)
        
        opt_student.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            s_logits, _ = student_model(ids, mask, chunk_size=16)
            loss = distillation_loss(s_logits, t_logits, lbls, alpha_kd=0.5, temperature=4.0)
            
        scaler_s.scale(loss).backward()
        scaler_s.unscale_(opt_student)
        nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
        scaler_s.step(opt_student)
        scaler_s.update()
        
        total_loss += loss.item()
        n_steps += 1
        
    avg_loss = total_loss / max(1, n_steps)
    
    # Dev Evaluation
    student_model.eval()
    dev_logits_s, dev_labels_s = [], []
    with torch.no_grad():
        for doc in dev_docs:
            ids = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            logits, _ = student_model(ids, mask, chunk_size=16)
            dev_logits_s.append(logits.cpu().numpy())
            dev_labels_s.append(doc['labels'].numpy())
            
    val_f1 = compute_macro_f1(np.concatenate(dev_labels_s), np.concatenate(dev_logits_s).argmax(-1), NUM_CLASSES)
    print(f"Epoch {epoch}/{EPOCHS_RUN3} [{time.time()-t0:.1f}s] - KD Loss: {avg_loss:.4f} | Dev Macro-F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1_kd:
        best_macro_f1_kd = val_f1
        best_logits_kd_dev = dev_logits_s
        torch.save(student_model.state_dict(), os.path.join(WORK_DIR, 'arch4_1_kd_best.pt'))

bias_kd, tuned_f1_kd = tune_class_bias(best_logits_kd_dev, [d['labels'].numpy() for d in dev_docs], NUM_CLASSES)
print(f"\n✓ Run 3 Dev Macro-F1: {best_macro_f1_kd:.4f} -> Post-Hoc Calibrated F1: {tuned_f1_kd:.4f}")

# Generate Test Predictions
print("Generating Run 3 test predictions...")
student_model.eval()
test_logits_kd = {}
preds_kd = {}
with torch.no_grad():
    for doc in test_cl_ds + test_it_ds:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        logits, _ = student_model(ids, mask, chunk_size=16)
        test_logits_kd[doc['doc_id']] = logits.cpu().numpy()
        calibrated_preds = (logits.cpu().numpy() + bias_kd).argmax(-1).tolist()
        preds_kd[doc['doc_id']] = [ID2LABEL[p] for p in calibrated_preds]

sub_kd_zip = save_submission_zip(preds_kd, "arch4_1_kd", test_docs)

probs_kd_test = {k: F.softmax(torch.tensor(v).float() + torch.tensor(bias_kd), dim=-1).numpy().tolist() for k, v in test_logits_kd.items()}
with open(os.path.join(WORK_DIR, 'probs_kd_test.json'), 'w') as f:
    json.dump(probs_kd_test, f)

del student_model, opt_student, student_bb
if 'teacher_model' in locals(): del teacher_model
if 'teacher_bb' in locals(): del teacher_bb
if 'scaler_kd' in locals(): del scaler_kd
free_mem()
print("Run 3 completed. Memory purged.")

## ====================================================================
## RUN 5: Architecture 4 with AdaLoRA (Adaptive Low-Rank Allocation)
## Backbone: `ALiBi + SwiGLU + RMSNorm` with Dynamic SVD Rank Pruning & BiLSTM+MHA Head
## ====================================================================


In [ ]:
# RUN 5: Architecture 4 + AdaLoRA Model Definition & Training
free_mem()

class AdaLoRALinear(nn.Module):
    def __init__(self, original: nn.Linear, init_rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.original = original
        self.rank = init_rank
        self.scaling = alpha / init_rank

        d_out = original.out_features
        d_in  = original.in_features

        self.lora_P = nn.Parameter(torch.empty(d_out, init_rank))
        self.lora_E = nn.Parameter(torch.zeros(init_rank))
        self.lora_Q = nn.Parameter(torch.empty(init_rank, d_in))
        self.drop   = nn.Dropout(dropout)

        self.register_buffer('importance', torch.zeros(init_rank))
        self.register_buffer('rank_mask', torch.ones(init_rank))

        nn.init.orthogonal_(self.lora_P)
        nn.init.orthogonal_(self.lora_Q)

        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

    def forward(self, x):
        masked_E = self.lora_E * self.rank_mask
        out = self.drop(x) @ self.lora_Q.t()
        out = out * masked_E
        out = out @ self.lora_P.t()
        return self.original(x) + out * self.scaling

    def update_importance(self, beta1=0.85):
        if self.lora_E.grad is not None:
            curr = (self.lora_E.abs() * self.lora_E.grad.abs()).detach()
            self.importance = beta1 * self.importance + (1 - beta1) * curr

class AdaLoRAManager:
    def __init__(self, model: nn.Module, target_rank: int = 8):
        self.target_rank = target_rank
        self.layers: List[AdaLoRALinear] = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        print(f"  AdaLoRA Manager initialized across {len(self.layers)} layers (target rank={target_rank}).")

    def update_importance(self):
        for layer in self.layers:
            layer.update_importance()

    def prune_ranks(self):
        all_imp = []
        for l in self.layers:
            for r in range(l.rank):
                all_imp.append((l.importance[r].item(), l, r))
        budget = self.target_rank * len(self.layers)
        all_imp.sort(key=lambda x: x[0], reverse=True)
        for l in self.layers:
            l.rank_mask.zero_()
        for _, layer, r_idx in all_imp[:budget]:
            layer.rank_mask[r_idx] = 1.0

def inject_adalora(model: nn.Module, targets: List[str], init_rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
    cnt = 0
    for name, module in list(model.named_modules()):
        for t in targets:
            if name.endswith(t) and isinstance(module, nn.Linear):
                p = model
                for part in name.split('.')[:-1]:
                    p = getattr(p, part)
                setattr(p, name.split('.')[-1], AdaLoRALinear(module, init_rank, alpha, dropout))
                cnt += 1
                break
    print(f"  AdaLoRA injected into {cnt} linear projections (init_rank={init_rank}).")

class RRClassifierArch4AdaLoRA(nn.Module):
    def __init__(self, backbone: ALiBiSwiGLULM, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model // 2,
            num_layers=2, batch_first=True, bidirectional=True, dropout=dropout
        )
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def get_sent_embs(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            c_ids = input_ids[i : i + chunk_size]
            c_mask = attention_mask[i : i + chunk_size]
            T = int(c_mask.sum(1).max().item())
            ci = c_ids[:, :max(T, 1)]
            cm = c_mask[:, :max(T, 1)]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                h = self.backbone(ci)
            m = cm.unsqueeze(-1).float()
            emb = (h.float() * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        return torch.cat(sent_embs, dim=0)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        embs = self.get_sent_embs(input_ids, attention_mask, chunk_size=chunk_size)
        x = embs.unsqueeze(0)
        lstm_out, _ = self.bilstm(x)
        lstm_out = self.drop(lstm_out.squeeze(0).float())
        attn_out, _ = self.attn(x, x, x)
        attn_out = self.drop(self.norm(attn_out).squeeze(0).float())
        combined = torch.cat([lstm_out, attn_out], dim=-1)
        return self.classifier(combined)

print("Initializing Architecture 4 with AdaLoRA...")
cfg_adalora = ALiBiConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=2560)
backbone_adalora = ALiBiSwiGLULM(cfg_adalora)

if ARCH4_CHECKPOINT_PATH and os.path.exists(ARCH4_CHECKPOINT_PATH):
    print(f"Loading pretrained weights from {ARCH4_CHECKPOINT_PATH}...")
    ckpt_ada = torch.load(ARCH4_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state_dict_ada = ckpt_ada.get('model_state_dict', ckpt_ada)
    backbone_adalora.load_state_dict(state_dict_ada, strict=False)
    print("✓ Loaded Architecture 4 pretrained weights for AdaLoRA.")

for p in backbone_adalora.parameters():
    p.requires_grad = False

TARGET_PROJECTIONS = ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'gate_proj', 'up_proj', 'down_proj']
inject_adalora(backbone_adalora, TARGET_PROJECTIONS, init_rank=16, alpha=32.0, dropout=0.05)

model_adalora = RRClassifierArch4AdaLoRA(backbone_adalora, NUM_CLASSES, dropout=0.1).to(device)
ada_manager = AdaLoRAManager(model_adalora, target_rank=8)

trainable_p_ada = sum(p.numel() for p in model_adalora.parameters() if p.requires_grad)
total_p_ada = sum(p.numel() for p in model_adalora.parameters())
print(f"Architecture 4 AdaLoRA Ready: {trainable_p_ada:,} / {total_p_ada:,} trainable parameters ({100*trainable_p_ada/total_p_ada:.2f}%)")

opt_adalora = AdamW([p for p in model_adalora.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
scaler_adalora = torch.amp.GradScaler('cuda', enabled=USE_AMP)
EPOCHS_RUN5 = 4
WARMUP_PRUNE_STEPS = 20
RANK_PRUNE_INTERVAL = 10

print("\nStarting Run 5 Training (Arch 4 AdaLoRA)...")
best_macro_f1_adalora = 0.0
best_logits_adalora_dev = []
global_step_ada = 0

for epoch in range(1, EPOCHS_RUN5 + 1):
    model_adalora.train()
    total_loss, n_steps = 0.0, 0
    t0 = time.time()
    
    perm = torch.randperm(len(train_docs)).tolist()
    for idx in perm:
        doc = train_docs[idx]
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        
        opt_adalora.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model_adalora(ids, mask, chunk_size=16)
            loss = criterion(logits, lbls)
            
        scaler_adalora.scale(loss).backward()
        ada_manager.update_importance()
        scaler_adalora.unscale_(opt_adalora)
        nn.utils.clip_grad_norm_([p for p in model_adalora.parameters() if p.requires_grad], max_norm=1.0)
        scaler_adalora.step(opt_adalora)
        scaler_adalora.update()
        
        global_step_ada += 1
        if global_step_ada > WARMUP_PRUNE_STEPS and global_step_ada % RANK_PRUNE_INTERVAL == 0:
            ada_manager.prune_ranks()
            
        total_loss += loss.item()
        n_steps += 1
        
    avg_loss = total_loss / max(1, n_steps)
    
    # Dev Evaluation
    model_adalora.eval()
    dev_logits_ada, dev_labels_ada = [], []
    with torch.no_grad():
        for doc in dev_docs:
            ids = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            logits = model_adalora(ids, mask, chunk_size=16)
            dev_logits_ada.append(logits.cpu().numpy())
            dev_labels_ada.append(doc['labels'].numpy())
            
    val_f1 = compute_macro_f1(np.concatenate(dev_labels_ada), np.concatenate(dev_logits_ada).argmax(-1), NUM_CLASSES)
    print(f"Epoch {epoch}/{EPOCHS_RUN5} [{time.time()-t0:.1f}s] - Train Loss: {avg_loss:.4f} | Dev Macro-F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1_adalora:
        best_macro_f1_adalora = val_f1
        best_logits_adalora_dev = dev_logits_ada
        torch.save(model_adalora.state_dict(), os.path.join(WORK_DIR, 'arch4_adalora_best.pt'))

bias_adalora, tuned_f1_adalora = tune_class_bias(best_logits_adalora_dev, [d['labels'].numpy() for d in dev_docs], NUM_CLASSES)
print(f"\n✓ Run 5 Dev Macro-F1: {best_macro_f1_adalora:.4f} -> Post-Hoc Calibrated F1: {tuned_f1_adalora:.4f}")

# Generate Test Predictions
print("Generating Run 5 test predictions...")
model_adalora.eval()
test_logits_adalora = {}
preds_adalora = {}
with torch.no_grad():
    for doc in test_cl_ds + test_it_ds:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        logits = model_adalora(ids, mask, chunk_size=16)
        test_logits_adalora[doc['doc_id']] = logits.cpu().numpy()
        calibrated_preds = (logits.cpu().numpy() + bias_adalora).argmax(-1).tolist()
        preds_adalora[doc['doc_id']] = [ID2LABEL[p] for p in calibrated_preds]

sub_arch4_adalora_zip = save_submission_zip(preds_adalora, "arch4_adalora", test_docs)

probs_adalora_test = {k: F.softmax(torch.tensor(v).float() + torch.tensor(bias_adalora), dim=-1).numpy().tolist() for k, v in test_logits_adalora.items()}
with open(os.path.join(WORK_DIR, 'probs_arch4_adalora_test.json'), 'w') as f:
    json.dump(probs_adalora_test, f)

del model_adalora, opt_adalora, ada_manager, backbone_adalora, scaler_adalora
free_mem()
print("Run 5 completed. Memory purged.")

## ====================================================================
## RUN 6: Architecture 4 with (IA)^3 & Linear-Chain CRF
## Backbone: `ALiBi + SwiGLU + RMSNorm` with Infused Adapter Scaling Vectors + CRF Head
## ====================================================================


In [ ]:
# RUN 6: Architecture 4 + (IA)^3 Model Definition & Training
free_mem()

class ALiBiCausalAttentionIA3(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.dh = d_model // n_heads; self.dropout = dropout
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.q_norm  = RMSNorm(self.dh); self.k_norm = RMSNorm(self.dh)
        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer('slopes', slopes)
        self.ia3_k = None
        self.ia3_v = None

    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k)
        if self.ia3_k is not None:
            k = k * self.ia3_k.view(1, self.n_heads, 1, self.dh)
        if self.ia3_v is not None:
            v = v * self.ia3_v.view(1, self.n_heads, 1, self.dh)
        pq = torch.arange(S, device=x.device).unsqueeze(1)
        pk = torch.arange(S, device=x.device).unsqueeze(0)
        alibi = self.slopes * (pk - pq).unsqueeze(0)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = alibi + causal.unsqueeze(0).unsqueeze(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1,2).contiguous().view(B, S, self.n_heads * self.dh))

class SwiGLUIA3(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        self.ia3_ff = None
    def forward(self, x):
        gate = F.silu(self.gate_proj(x)) * self.up_proj(x)
        if self.ia3_ff is not None:
            gate = gate * self.ia3_ff.unsqueeze(0).unsqueeze(0)
        return self.drop(self.down_proj(gate))

class ALiBiBlockIA3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1   = RMSNorm(cfg.d_model)
        self.attn = ALiBiCausalAttentionIA3(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.n2   = RMSNorm(cfg.d_model)
        self.ffn  = SwiGLUIA3(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x

class ALiBiSwiGLULMIA3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([ALiBiBlockIA3(cfg) for _ in range(cfg.n_layers)])
        self.norm_f  = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks: x = b(x)
        return self.norm_f(x)

class IA3Module(nn.Module):
    def __init__(self, backbone: ALiBiSwiGLULMIA3):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        d_ff    = backbone.cfg.d_ff
        L       = backbone.cfg.n_layers

        self.l_k  = nn.ParameterList([nn.Parameter(torch.ones(d_model)) for _ in range(L)])
        self.l_v  = nn.ParameterList([nn.Parameter(torch.ones(d_model)) for _ in range(L)])
        self.l_ff = nn.ParameterList([nn.Parameter(torch.ones(d_ff)) for _ in range(L)])

    def attach_scales(self):
        for i, block in enumerate(self.backbone.blocks):
            block.attn.ia3_k = self.l_k[i]
            block.attn.ia3_v = self.l_v[i]
            block.ffn.ia3_ff = self.l_ff[i]

    def forward(self, idx):
        self.attach_scales()
        return self.backbone(idx)

class CRF(nn.Module):
    def __init__(self, num_tags: int):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions   = nn.Parameter(torch.empty(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def neg_log_likelihood(self, emissions: torch.Tensor, tags: torch.Tensor) -> torch.Tensor:
        emissions = emissions.float()
        S = emissions.size(0)
        score = self.start_transitions[tags[0]] + emissions[0, tags[0]]
        for i in range(1, S):
            score += self.transitions[tags[i], tags[i-1]] + emissions[i, tags[i]]
        score += self.end_transitions[tags[-1]]
        alpha = self.start_transitions + emissions[0]
        for i in range(1, S):
            alpha = torch.logsumexp(alpha.unsqueeze(0) + self.transitions + emissions[i].unsqueeze(1), dim=1)
        log_Z = torch.logsumexp(alpha + self.end_transitions, dim=0)
        return log_Z - score

    def decode(self, emissions: torch.Tensor) -> List[int]:
        emissions = emissions.float()
        S, K = emissions.shape
        viterbi = torch.full((S, K), -1e9, device=emissions.device)
        backptr = torch.zeros((S, K), dtype=torch.long, device=emissions.device)
        viterbi[0] = self.start_transitions + emissions[0]
        for i in range(1, S):
            trans = viterbi[i-1].unsqueeze(1) + self.transitions
            viterbi[i], backptr[i] = trans.max(dim=0)
            viterbi[i] += emissions[i]
        viterbi[-1] += self.end_transitions
        best = [viterbi[-1].argmax().item()]
        for i in range(S-1, 0, -1):
            best.append(backptr[i, best[-1]].item())
        return list(reversed(best))

class IA3RRModel(nn.Module):
    def __init__(self, ia3: IA3Module, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.ia3 = ia3
        d_model = ia3.backbone.cfg.d_model
        self.bilstm = nn.LSTM(d_model, d_model // 2, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.emission_head = nn.Linear(d_model, num_labels)
        self.crf = CRF(num_labels)

    def get_sent_embs(self, input_ids: torch.Tensor, attn_mask: torch.Tensor, chunk_size: int = 16):
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            c_ids = input_ids[i : i + chunk_size]
            c_mask = attn_mask[i : i + chunk_size]
            T = int(c_mask.sum(1).max().item())
            ci = c_ids[:, :max(T, 1)]
            cm = c_mask[:, :max(T, 1)]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                h = self.ia3(ci)
            m = cm.unsqueeze(-1).float()
            emb = (h.float() * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        return torch.cat(sent_embs, dim=0)

    def forward(self, input_ids, attn_mask, labels=None, chunk_size=16):
        embs = self.get_sent_embs(input_ids, attn_mask, chunk_size=chunk_size)
        x = embs.unsqueeze(0)
        lstm_out, _ = self.bilstm(x)
        emissions = self.emission_head(self.drop(lstm_out.squeeze(0).float()))
        if labels is not None:
            valid = labels >= 0
            if valid.sum() == 0:
                loss = torch.tensor(0.0, requires_grad=True, device=input_ids.device)
            else:
                loss = self.crf.neg_log_likelihood(emissions[valid], labels[valid])
            return loss, emissions
        else:
            preds = self.crf.decode(emissions)
            return preds, emissions

print("Initializing Architecture 4 with (IA)^3 + CRF...")
cfg_ia3 = ALiBiConfig(vocab_size=tokenizer.vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=2560)
backbone_ia3 = ALiBiSwiGLULMIA3(cfg_ia3)

if ARCH4_CHECKPOINT_PATH and os.path.exists(ARCH4_CHECKPOINT_PATH):
    print(f"Loading pretrained weights from {ARCH4_CHECKPOINT_PATH}...")
    ckpt_ia3 = torch.load(ARCH4_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state_dict_ia3 = ckpt_ia3.get('model_state_dict', ckpt_ia3)
    backbone_ia3.load_state_dict(state_dict_ia3, strict=False)
    print("✓ Loaded Architecture 4 pretrained weights for (IA)^3.")

for p in backbone_ia3.parameters():
    p.requires_grad = False

ia3_module = IA3Module(backbone_ia3)
model_ia3 = IA3RRModel(ia3_module, NUM_CLASSES, dropout=0.1).to(device)

trainable_p_ia3 = sum(p.numel() for p in model_ia3.parameters() if p.requires_grad)
total_p_ia3 = sum(p.numel() for p in model_ia3.parameters())
print(f"Architecture 4 (IA)^3 Ready: {trainable_p_ia3:,} / {total_p_ia3:,} trainable parameters ({100*trainable_p_ia3/total_p_ia3:.2f}%)")

opt_ia3 = AdamW([p for p in model_ia3.parameters() if p.requires_grad], lr=5e-4, weight_decay=0.01)
scaler_ia3 = torch.amp.GradScaler('cuda', enabled=USE_AMP)
EPOCHS_RUN6 = 4

print("\nStarting Run 6 Training (Arch 4 IA3 + CRF)...")
best_macro_f1_ia3 = 0.0
best_emissions_ia3_dev = []

for epoch in range(1, EPOCHS_RUN6 + 1):
    model_ia3.train()
    total_loss, n_steps = 0.0, 0
    t0 = time.time()
    
    perm = torch.randperm(len(train_docs)).tolist()
    for idx in perm:
        doc = train_docs[idx]
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        
        opt_ia3.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            loss, _ = model_ia3(ids, mask, labels=lbls, chunk_size=16)
            
        scaler_ia3.scale(loss).backward()
        scaler_ia3.unscale_(opt_ia3)
        nn.utils.clip_grad_norm_([p for p in model_ia3.parameters() if p.requires_grad], max_norm=1.0)
        scaler_ia3.step(opt_ia3)
        scaler_ia3.update()
        
        total_loss += loss.item()
        n_steps += 1
        
    avg_loss = total_loss / max(1, n_steps)
    
    # Dev Evaluation
    model_ia3.eval()
    dev_preds_ia3, dev_emissions_ia3, dev_labels_ia3 = [], [], []
    with torch.no_grad():
        for doc in dev_docs:
            ids = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            preds, emissions = model_ia3(ids, mask, labels=None, chunk_size=16)
            dev_preds_ia3.append(np.array(preds))
            dev_emissions_ia3.append(emissions.cpu().numpy())
            dev_labels_ia3.append(doc['labels'].numpy())
            
    val_f1 = compute_macro_f1(np.concatenate(dev_labels_ia3), np.concatenate(dev_preds_ia3), NUM_CLASSES)
    print(f"Epoch {epoch}/{EPOCHS_RUN6} [{time.time()-t0:.1f}s] - Train CRF Loss: {avg_loss:.4f} | Dev Macro-F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1_ia3:
        best_macro_f1_ia3 = val_f1
        best_emissions_ia3_dev = dev_emissions_ia3
        torch.save(model_ia3.state_dict(), os.path.join(WORK_DIR, 'arch4_ia3_best.pt'))

print(f"\n✓ Run 6 Peak Dev Macro-F1: {best_macro_f1_ia3:.4f}")
tuned_f1_ia3 = best_macro_f1_ia3

# Generate Test Predictions
print("Generating Run 6 test predictions...")
model_ia3.eval()
test_emissions_ia3 = {}
preds_ia3 = {}
with torch.no_grad():
    for doc in test_cl_ds + test_it_ds:
        ids = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        preds, emissions = model_ia3(ids, mask, labels=None, chunk_size=16)
        preds_ia3[doc['doc_id']] = [ID2LABEL[p] for p in preds]
        test_emissions_ia3[doc['doc_id']] = emissions.cpu().numpy()

sub_arch4_ia3_zip = save_submission_zip(preds_ia3, "arch4_ia3", test_docs)

probs_ia3_test = {k: F.softmax(torch.tensor(v).float(), dim=-1).numpy().tolist() for k, v in test_emissions_ia3.items()}
with open(os.path.join(WORK_DIR, 'probs_arch4_ia3_test.json'), 'w') as f:
    json.dump(probs_ia3_test, f)

del model_ia3, opt_ia3, ia3_module, backbone_ia3, scaler_ia3
free_mem()
print("Run 6 completed. Memory purged.")

## ====================================================================
## MASTER ENSEMBLE: Multi-Model Soft Voting across All Completed Runs
## Dynamic Weighted Combination of Run 1 (LORMA), Run 2 (Prefix), Run 3 (KD), Run 5 (AdaLoRA) & Run 6 (IA3)
## ====================================================================


In [ ]:
# MASTER ENSEMBLE: Dynamic Multi-Model Weighted Voting & Submission
print("Computing Dynamic Multi-Model Ensemble across Completed Runs...")
gold_dev_labels = np.concatenate([d['labels'].numpy() for d in dev_docs])

model_runs = {}

if 'best_logits_arch4_dev' in globals() and 'probs_arch4_test' in globals():
    p4 = F.softmax(torch.tensor(np.concatenate(best_logits_arch4_dev) + bias_arch4).float(), dim=-1).numpy()
    model_runs['Run 1 (Arch 4 LORMA)'] = {
        'dev_probs': p4,
        'dev_f1': tuned_f1_arch4,
        'probs_test': probs_arch4_test,
        'weight': 0.30
    }

if 'best_logits_arch3_dev' in globals() and 'probs_arch3_test' in globals():
    p3 = F.softmax(torch.tensor(np.concatenate(best_logits_arch3_dev) + bias_arch3).float(), dim=-1).numpy()
    model_runs['Run 2 (Arch 3 Prefix Tuning)'] = {
        'dev_probs': p3,
        'dev_f1': tuned_f1_arch3,
        'probs_test': probs_arch3_test,
        'weight': 0.15
    }

if 'best_logits_kd_dev' in globals() and 'probs_kd_test' in globals():
    pk = F.softmax(torch.tensor(np.concatenate(best_logits_kd_dev) + bias_kd).float(), dim=-1).numpy()
    model_runs['Run 3 (Arch 4->1 KD + LoRA)'] = {
        'dev_probs': pk,
        'dev_f1': tuned_f1_kd,
        'probs_test': probs_kd_test,
        'weight': 0.15
    }

if 'best_logits_adalora_dev' in globals() and 'probs_adalora_test' in globals():
    pa = F.softmax(torch.tensor(np.concatenate(best_logits_adalora_dev) + bias_adalora).float(), dim=-1).numpy()
    model_runs['Run 5 (Arch 4 AdaLoRA)'] = {
        'dev_probs': pa,
        'dev_f1': tuned_f1_adalora,
        'probs_test': probs_adalora_test,
        'weight': 0.25
    }

if 'best_emissions_ia3_dev' in globals() and 'probs_ia3_test' in globals():
    pi = F.softmax(torch.tensor(np.concatenate(best_emissions_ia3_dev)).float(), dim=-1).numpy()
    model_runs['Run 6 (Arch 4 IA3 + CRF)'] = {
        'dev_probs': pi,
        'dev_f1': tuned_f1_ia3,
        'probs_test': probs_ia3_test,
        'weight': 0.15
    }

if not model_runs:
    raise RuntimeError("No completed runs found to ensemble! Please execute at least one experiment run first.")

# Normalize weights across available runs
total_w = sum(info['weight'] for info in model_runs.values())
for k in model_runs:
    model_runs[k]['norm_weight'] = model_runs[k]['weight'] / total_w

# Compute weighted ensemble on dev
ens_dev_probs = sum(info['norm_weight'] * info['dev_probs'] for info in model_runs.values())
ens_dev_preds = ens_dev_probs.argmax(axis=-1)
ens_macro_f1 = compute_macro_f1(gold_dev_labels, ens_dev_preds, NUM_CLASSES)

print("=" * 75)
print("FINAL MACRO-F1 PERFORMANCE SUMMARY (DEV SET):")
print("=" * 75)
for name, info in model_runs.items():
    print(f"  {name:<36} (w={info['norm_weight']:.2f}) : {info['dev_f1']:.4f}")
print("-" * 75)
print(f"  {'Master Multi-Model Ensemble':<36} (Overall) : {ens_macro_f1:.4f}  <-- HIGHEST SCORE")
print("=" * 75)

# Synthesize Ensemble Test Predictions
print("\nSynthesizing Master Ensemble Test Predictions...")
final_ensemble_preds = {}
test_doc_ids = [d['id'] for d in test_docs]

for d_id in test_doc_ids:
    doc_ens_prob = sum(info['norm_weight'] * np.array(info['probs_test'][d_id]) for info in model_runs.values())
    pred_indices = doc_ens_prob.argmax(axis=-1).tolist()
    final_ensemble_preds[d_id] = [ID2LABEL[idx] for idx in pred_indices]

sub_ensemble_zip = save_submission_zip(final_ensemble_preds, "ensemble", test_docs)
print(f"✓ Master Ensemble Submission Ready: {sub_ensemble_zip}")

In [ ]:
# Verification & Downloadable Links for All Submissions
from IPython.display import FileLink, display

print("\n" + "=" * 70)
print("SUBMISSION PACKAGES READY FOR CODABENCH:")
print("=" * 70)

submission_files = [
    os.path.join(WORK_DIR, "submission_arch4_lorma.zip"),
    os.path.join(WORK_DIR, "submission_arch3_prefixtuning.zip"),
    os.path.join(WORK_DIR, "submission_arch4_1_kd.zip"),
    os.path.join(WORK_DIR, "submission_arch4_adalora.zip"),
    os.path.join(WORK_DIR, "submission_arch4_ia3.zip"),
    os.path.join(WORK_DIR, "submission_ensemble.zip"),
]

for s_path in submission_files:
    if os.path.exists(s_path):
        size_kb = os.path.getsize(s_path) / 1024
        with zipfile.ZipFile(s_path, 'r') as z:
            contents = z.namelist()
            pred_data = json.loads(z.read("prediction.json").decode('utf-8'))
            n_docs = len(pred_data)
            n_sents = sum(len(v) for v in pred_data.values())
        print(f"• File: {os.path.basename(s_path):<32} | Size: {size_kb:>6.1f} KB | Docs: {n_docs} | Sents: {n_sents}")
        print(f"  Archive contains: {contents}")
        try:
            display(FileLink(s_path))
        except Exception:
            pass
    else:
        print(f"• Missing: {s_path}")
print("=" * 70)
print("All tasks completed successfully!")